## 10. Altering Tables and Complex Table Relationships: ALTER TABLE, ADD COLUMN, DROP COLUMN, RENAME TO, RENAME COLUMN, ALTER COLUMN, SET NOT NULL, DROP NOT NULL, FOREIGN KEY

In this notebook different ways to alter a table are covered, as well as advanced relationships like FOREIGN KEYs and Many to Many relationship tables.

###### Keywords Covered - ALTER TABLE, ADD COLUMN, DROP COLUMN, RENAME TO, RENAME COLUMN, ALTER COLUMN, SET NOT NULL, DROP NOT NULL, FOREIGN KEY

In [1]:
import sqlite3
import pandas as pd

# Connect to database
data_conn = sqlite3.connect("pokemon.db")

In [2]:
# Create a table
data_conn.execute("""CREATE TABLE pokemon_moves (name TEXT NOT NULL, Type TEXT NOT NULL, damage INTEGER NOT NULL)""")

# Insert data
data_conn.execute("""INSERT INTO pokemon_moves (name, type, damage) VALUES
                ('Fire Blast', 'Fire', 120), ('Surf', 'Water', 85),
                ('Tackle', 'Normal', 20), ('Ice Beam', 'Ice', 60)""")

In [3]:
# Display the data in the table
pd.read_sql("""SELECT * FROM pokemon_moves""", con=data_conn)

,name,Type,damage
0,Fire Blast,Fire,120
1,Surf,Water,85
2,Tackle,Normal,20
3,Ice Beam,Ice,60


### ALTER TABLE

Use: Modifying a table in combination with other keywords.

### ADD COLUMN

Use: Adding a column to the tabel. Adds it as the last column.

### DROP COLUMN

Use: Deleting a column.

### RENAME TO

Use: Renaming a table.

*Syntax: ALTER TABLE pokemone_moves RENAME TO poke_moves*

### RENAME COLUMN

Use: Renaming a column.

*Syntax: RENAME COLUMN power TO damage*

Below we will modify the table pokemon_moves in a variety of ways and show the results.

In [4]:
# Add a column for the effect as a string
data_conn.execute("""ALTER TABLE pokemon_moves
                     ADD COLUMN effect VARCHAR""")

# Drop the damage column
data_conn.execute("""ALTER TABLE pokemon_moves
                     DROP COLUMN damage""")

# Change the table name
data_conn.execute("""ALTER TABLE pokemon_moves
                     RENAME TO poke_moves""")

### Commit()

Use: Commiting the changes to  the database.

*Tip: make sure to commit changes or modifications may not be saved.*

In [5]:
# Commit the changes
data_conn.commit()

# Display the data in the table
pd.read_sql("""SELECT * FROM poke_moves""", con=data_conn)

,name,Type,effect
0,Fire Blast,Fire,None
1,Surf,Water,None
2,Tackle,Normal,None
3,Ice Beam,Ice,None


### ALTER COLUMN

You can also set or drop the NOT NULL constraint in a table with ALTER COLUMN.

### SET NOT NULL

Use: Setting the NOT NULL constraint to a column.

*Syntax: ALTER COLUMN column_name SET NOT NULL*

### DROP NOT NULL

Use: Dropping the NOT NULL constraint to a column.

*Syntax: ALTER COLUMN column_name DROP NOT NULL*

Note: Not demonstrated as not available in SQLite. You must remake the table with the column having the NOT NULL restraint as required and then transfer the data accross.

### FOREIGN KEY

Use: 

FOREIGN KEY (constraint) or item__owner INT REFERENCES pokemon(id)

In [6]:
# Turn on foreighn key enforcement.
data_conn.execute("PRAGMA foreign_keys = ON;")

# Create a new moves table where the foreign key references the id column in the pokemon table.
data_conn.execute("""CREATE TABLE poke_moves_table(move_id INTEGER PRIMARY KEY, 
                                                  name TEXT NOT NULL,
                                                  pokemon_id INTEGER,

                                                  FOREIGN KEY (pokemon_id) REFERENCES pokemon (id))""")

In [7]:
# Inserting correct data into the table
data_conn.execute("""INSERT INTO poke_moves_table VALUES
                     (1, 'Fire Blast', 6),
                     (2, 'Surf', 9)""")

In [8]:
# Display the data in the table
pd.read_sql("""SELECT * FROM poke_moves_table""", con=data_conn)

,move_id,name,pokemon_id
0,1,Fire Blast,6
1,2,Surf,9


In [9]:
# Inserting incorrect data into the table throws an error as the FOREIGN KEY is not in the pokemon table
data_conn.execute("""INSERT INTO poke_moves_table VALUES
                     (3, 'Hyper Beam', 9999)""")

IntegrityError: FOREIGN KEY constraint failed

### Many to Many relationships

Use: Many to many relationship tables are used to define relationships between objects. For instance many gym owners have many Pokemon. So creating a table that links the gym_leaders table and the pokemon table could be used. This would show which gym leaders own which Pokemon.

In [10]:
# Many to many relationship table to asign moves to multiple Pokemon
data_conn.execute("""CREATE TABLE IF NOT EXISTS moves_users (
                        move_user INT REFERENCES pokemon (id),
                        move INT REFERENCES poke_moves_table (move_id),
                        PRIMARY KEY (move_user, move))""")

In [11]:
# Inserting demo data
data_conn.execute("""INSERT INTO moves_users VALUES
                     (6, 1)""")

In [12]:
# Display the data in the table
pd.read_sql("""SELECT * FROM moves_users""", con=data_conn)

,move_user,move
0,6,1


In [14]:
# Join the pokemon and pokemon_moves tables via the moves_users table
pd.read_sql("""SELECT p.name, p.type1, m.name
               FROM moves_users
               JOIN poke_moves_table m ON m.move_id = moves_users.move
               JOIN pokemon p ON p.id = moves_users.move_user""", con=data_conn)

,name,type1,name
0,Charizard,fire,Fire Blast


In [20]:
# Clean up and close the connection
data_conn.execute("""DROP TABLE IF EXISTS pokemon_moves""")
data_conn.execute("""DROP TABLE IF EXISTS poke_moves""")
data_conn.execute("""DROP TABLE IF EXISTS poke_moves_table""")
data_conn.execute("""DROP TABLE IF EXISTS moves_users""")
data_conn.commit()
data_conn.close()